# PL5 — Exercise 1: Corrupting the input for a masked language model
### Generative AI · MSc in Artificial Intelligence · Lab 5

BERT is trained by **hiding characters and asking the model to recover them**. In this exercise you build that corruption step by step (no model and no training needed).

| Step | What you do | Time |
|---|---|---|
| A | choose 15% of the positions | 3 min |
| B | split them 80% `[MASK]` / 10% random / 10% unchanged | 4 min |
| C | put it into a function that also returns the targets | 4 min |
| Q | two short questions | 4 min |

**Instructions.** Run the cells in order. Replace each `____` with **one line** of code, then run the `CHECK` cell right after it: ✅ means done. Stuck? Open the 💡 **Hint** — after trying for a minute.

**Names / student numbers:** _…_

In [1]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.14.1+cpu | device: cpu


### The data (nothing to do — just run)
Character-level Tiny Shakespeare, as in PL1. We add **one extra token id** for `[MASK]`; when we print text, `[MASK]` is shown as `▁`.

In [2]:
with open("data/tiny_shakespeare.txt", encoding="utf-8") as f:
    text = f.read()
chars = sorted(set(text))
MASK = len(chars)                      # one extra id for [MASK]
V = len(chars) + 1
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] if i < len(chars) else "▁" for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
CTX, BATCH = 128, 32
print(f"V = {V} (including [MASK] = {MASK}), {len(train_data):,} train characters")

def get_batch(split, batch=BATCH, context=CTX):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - context - 1, (batch,))
    x = torch.stack([src[i:i + context] for i in ix])
    y = torch.stack([src[i + 1:i + context + 1] for i in ix])   # y = x shifted one to the left
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x[0]:", repr(decode(xb[0, :50].tolist())))
print("y[0]:", repr(decode(yb[0, :50].tolist())), " <- the same text, one character later")

V = 66 (including [MASK] = 65), 1,003,854 train characters
x[0]: 'aith of heart\nTo his most royal person, hither com'
y[0]: 'ith of heart\nTo his most royal person, hither come'  <- the same text, one character later


---
## Exercise 1.1 — Corrupting the input for a masked LM
BERT's recipe (lecture, slide 9): choose **15%** of the positions; of those, **80%** become `[MASK]`, **10%** a random token, **10%** stay unchanged. The model must predict the **original** token at every chosen position.

We build it in three small steps on one batch, then put the pieces into a function.

#### Step A — choose the positions
`chosen` must be a boolean tensor with the same shape as `xb`, `True` with probability `p`.


<details><summary>💡 <b>Hint</b> (try first!)</summary>

`torch.rand(shape)` gives uniform numbers in [0, 1). How often is such a number smaller than `p`? Use `device=xb.device` so it lives on the same device as the data.

</details>

In [3]:
torch.manual_seed(0)
xb, _ = get_batch("train")
p = 0.15

chosen = torch.rand(xb.shape, device=xb.device) < p  
print(f"chosen {chosen.float().mean().item():.1%} of {xb.numel():,} positions")

chosen 14.9% of 4,096 positions


In [4]:
# CHECK 1.1 A
assert chosen.dtype == torch.bool, "chosen must be boolean (use a comparison like  < p)"
assert chosen.shape == xb.shape, "chosen must have the same shape as xb"
assert 0.11 < chosen.float().mean().item() < 0.19, "about 15% of positions should be True"
print("✅ Step A OK")

✅ Step A OK


#### Step B — decide what happens to each chosen position
We draw **one more** random number `r` per position and split the chosen ones by its value:

```
 r:  0 ─────────────── 0.8 ──── 0.9 ──── 1
     [MASK]  (80%)      random   unchanged
```
`to_mask` is given. Write `to_random`.


<details><summary>💡 <b>Hint</b> (try first!)</summary>

Combine three conditions with `&`: the position must be chosen, **and** `r >= 0.8`, **and** `r < 0.9`. Put each comparison in parentheses.

</details>

In [5]:
r = torch.rand(xb.shape, device=xb.device)    # one random number per position
to_mask   = chosen & (r < 0.8)                 # given: 80% of the chosen -> [MASK]
to_random = chosen & (r >= 0.8) & (r < 0.9)      # chosen AND in the slice [0.8, 0.9)
# the last 10% of the chosen (r >= 0.9) stay unchanged — no code needed

n = chosen.sum().item()
print(f"of {n} chosen: {to_mask.sum().item()} -> [MASK], {to_random.sum().item()} -> random")

of 610 chosen: 479 -> [MASK], 65 -> random


In [6]:
# CHECK 1.1 B
assert not (to_random & ~chosen).any(), "to_random must only contain CHOSEN positions (did you forget  chosen & ...?)"
assert not (to_random & to_mask).any(), "a position cannot be both masked and random"
frac = to_random.sum().item() / chosen.sum().item()
assert 0.04 < frac < 0.17, f"about 10% of the chosen positions should be random, got {frac:.0%}"
print("✅ Step B OK")

✅ Step B OK


#### Step C — put it together
The function below repeats steps A and B. Fill in the **targets**: at the chosen positions they must hold the **original** token; everywhere else they stay `-100`, which `F.cross_entropy` ignores.


<details><summary>💡 <b>Hint</b> (try first!)</summary>

Boolean indexing works on both sides: `targets[chosen] = x[chosen]` copies the values of `x` at the chosen positions. Note that this line comes **before** `x` is modified.

</details>

In [7]:
def mlm_batch(x, p=0.15):
    x = x.clone()                                   # never modify the caller's tensor
    targets = torch.full_like(x, -100)              # -100 everywhere = "no loss here"
    chosen = torch.rand(x.shape, device=x.device) < p
    r = torch.rand(x.shape, device=x.device)
    to_mask = chosen & (r < 0.8)
    to_random = chosen & (r >= 0.8) & (r < 0.9)

    targets[chosen] = x[chosen]                   # ORIGINAL tokens, copied BEFORE x is corrupted
    x[to_mask] = MASK
    x[to_random] = torch.randint(0, MASK, (int(to_random.sum()),), device=x.device)
    return x, targets

In [8]:
# CHECK 1.1 (same checks as the original lab)
torch.manual_seed(0)
xb, _ = get_batch("train")
xc, tg = mlm_batch(xb, p=0.15)
sel = tg != -100
frac = sel.float().mean().item()
assert 0.11 < frac < 0.19, frac
assert torch.equal(tg[sel], xb[sel]), "targets must hold the ORIGINAL tokens"
assert torch.equal(xb[~sel], xc[~sel]), "positions that were not chosen must not change"
masked_frac = (xc[sel] == MASK).float().mean().item()
assert 0.7 < masked_frac < 0.9, masked_frac
unchanged = (xc[sel] == xb[sel]).float().mean().item()
print(f"✅ 1.1 OK — chosen {frac:.1%}, of those [MASK] {masked_frac:.0%}, left unchanged {unchanged:.0%}")

# Look at what the encoder will be trained on (▁ = [MASK], ^ = a chosen position)
print("\noriginal :", repr(decode(xb[0, :60].tolist())))
print("corrupted:", repr(decode(xc[0, :60].tolist())))
print("chosen   :  " + "".join("^" if s else " " for s in sel[0, :60].tolist()))

✅ 1.1 OK — chosen 14.9%, of those [MASK] 79%, left unchanged 11%

original : 'aith of heart\nTo his most royal person, hither come\nEven at '
corrupted: 'ait▁ of hea▁▁\nTo his most▁royal person▁ hithe▁ c▁me▁Even at '
chosen   :     ^       ^^            ^            ^      ^  ^  ^        


---
## Questions

**Q1.** Why are only **80%** of the chosen positions replaced by `[MASK]`, and not 100%? *(hint: does `[MASK]` ever appear when the model is used after training?)*

**Q2.** Some chosen positions are left **unchanged**, yet they still have a target. What does this force the model to do with *every* position, not only the masked ones?

**Your answers:**

Q1: The ``,` symbol never appears in the input when the model is *used* after pretraining — during fine-tuning or inference the encoder always sees real text. If 100% of the chosen positions were replaced by ``,`, the model would only ever learn to make predictions at positions marked with a special token it will never see again; that is a distribution mismatch between pretraining and actual use. Keeping 80% ``,`, 10% random and 10% unchanged means the model is also trained to predict tokens from *real* (possibly wrong!) tokens in context, so its representations stay useful for natural text, not just for inputs sprinkled with ``,` placeholders.

Q2: At run time the model has no way of knowing which positions were chosen, so to reduce the loss it must produce a good prediction of the current token at *every* position, not just at ``,` positions. The unchanged-but-targeted positions in particular force it to keep a sharp, accurate representation of tokens it can already see (it cannot simply learn "copy the input wherever it isn't ``,`"), which is exactly the rich contextual representation that downstream/fine-tuning tasks build on.